# Combined Recommendation System

combining multiple recommenders with weighted scores. easy to add new ones later

In [1]:
%load_ext autoreload
%autoreload 2

## setup and data loading

first connect to db and load all the data we need

In [1]:
import os
import asyncio
import asyncpg
from dotenv import load_dotenv
import pandas as pd
from urllib.parse import urlparse, parse_qs, urlencode, urlunparse

load_dotenv()
DATABASE_URL = os.getenv('DATABASE_URL')
if (DATABASE_URL is None):
    raise ValueError("DATABASE_URL is not set in environment variables")


def remove_schema_param(url):
    parsed = urlparse(url)
    query_params = parse_qs(parsed.query)

    query_params.pop('schema', None)

    new_query = urlencode(query_params, doseq=True)
    new_parsed = parsed._replace(query=new_query)
    return urlunparse(new_parsed)

asyncpg_url = remove_schema_param(DATABASE_URL)
conn = await asyncpg.connect(dsn=asyncpg_url)

In [2]:
from collaborative_filtering.utils import load_trade_tags

if not os.path.exists('models'):
    os.makedirs('models')

print("Loading Trade Data ")
user_trades_df = load_trade_tags("collaborative_filtering/df_trade_tags_cache")

len(user_trades_df)

Loading Trade Data 
Loading from 142 parquet files in 'collaborative_filtering/df_trade_tags_cache/'...


14259585

In [3]:
# Perform 80-20 temporal split per User
user_trades_df = user_trades_df.sort_values(['user_id', 'timestamp'])
user_trades_df['row_num'] = user_trades_df.groupby('user_id').cumcount() #assigns 0, 1, 2, 3... 
user_trades_df['total'] = user_trades_df.groupby('user_id')['timestamp'].transform('size') #how many trades for each user
user_trades_df['cutoff_idx'] = (user_trades_df['total'] * 0.8).astype(int) 
user_trades_df['cutoff_idx'] = user_trades_df['cutoff_idx'].clip(lower=1, upper=user_trades_df['total']-1) # where 80% falls

#filter users with at least 2 trades
user_trades_df = user_trades_df[user_trades_df['total'] >= 2]

#20% of each user's trades
test_df = user_trades_df[user_trades_df['row_num'] >= user_trades_df['cutoff_idx']].copy()
#timestamp slit per user
cutoff_map = test_df.groupby('user_id')['timestamp'].min().to_dict()

print(f"Created cutoff map for {len(cutoff_map)} users")

Created cutoff map for 8579 users


In [ ]:
# extract unique trades
raw_trades = user_trades_df[['user_id', 'item', 'timestamp', 'event_id']].drop_duplicates()
raw_trades.rename(columns={'user_id': 'address'}, inplace=True)

raw_trades['timestamp'] = pd.to_datetime(raw_trades['timestamp'])
raw_trades['event_id'] = raw_trades['event_id'].astype('Int64').astype(str)

print(f"Extracted {len(raw_trades):,} raw trades.")

# map items back to event IDs
item_to_event_id = raw_trades.dropna(subset=['item', 'event_id']).set_index('item')['event_id'].to_dict()

# maps each user cutoff timestamp to the trades
raw_trades['cutoff_ts'] = raw_trades['address'].map(cutoff_map)

#split by timestamp
initial_train_data = raw_trades[raw_trades['timestamp'] < raw_trades['cutoff_ts']].copy()
initial_test_data = raw_trades[raw_trades['timestamp'] >= raw_trades['cutoff_ts']].copy()

initial_train_data.drop(columns=['cutoff_ts'], inplace=True)
initial_test_data.drop(columns=['cutoff_ts'], inplace=True)

Extracted 2,277,933 raw trades.


setup

load the FAISS index and embeddings for finding similar events

In [17]:
# Topic-based setup
import faiss
import numpy as np
from bertopic import BERTopic
from sentence_transformers import SentenceTransformer
import pickle
import math

from utils import load_events_df, load_user_event_associations

index_path = "event_embeddings_shard_"
meta_path = "models/event_index_meta.pkl"
model_name_label = "all-MiniLM-L6-v2"

async def setup_topic_model():
    # conn = await asyncpg.connect(dsn=os.getenv('DATABASE_URL').replace('?schema=public', ''))
    conn = None # Skip DB connection, rely on cache
    events_df = await load_events_df(conn)
    user_event_associations_df = await load_user_event_associations(conn)
    # await conn.close()
    
    # load faiss
    if os.path.exists(meta_path):
        with open(meta_path, "rb") as f:
            meta = pickle.load(f)
        index_to_id = meta["index_to_id"]
        id_to_index = meta["id_to_index"]
        
        indices = []
        shard_paths = sorted(p for p in os.listdir("models") if p.startswith("event_embeddings_shard_"))
        for p in shard_paths:
            indices.append(faiss.read_index(os.path.join("models", p)))
            
        d = indices[0].d
        index = faiss.IndexFlatIP(d)
        for idx in indices:
            # rebuild index
            n = idx.ntotal
            if hasattr(idx, "reconstruct_n"):
                xb = idx.reconstruct_n(0, n)
            else:
                xb = np.vstack([idx.reconstruct(i) for i in range(n)])
            index.add(xb.astype("float32"))
            
        if hasattr(index, "reconstruct_n"):
            normalized_embeddings = index.reconstruct_n(0, index.ntotal)
        else:
             normalized_embeddings = np.vstack([index.reconstruct(i) for i in range(index.ntotal)])
             
    else:
        print("building indexes...")
        topic_model = BERTopic(embedding_model=model_name_label)

        # Create combined text for embeddings
        events_df["combined_text"] = (
            events_df["title"].fillna("") + " " + events_df["description"].fillna("")
        )

        embedding_model = SentenceTransformer(model_name_label)
        embeddings = embedding_model.encode(
            events_df["combined_text"].tolist(), show_progress_bar=True
        )

        embedding_dim = embeddings.shape[1]
        index = faiss.IndexFlatIP(embedding_dim)

        normalized_embeddings = embeddings / np.linalg.norm(
            embeddings, axis=1, keepdims=True
        )
        index.add(normalized_embeddings.astype("float32"))

        index_to_id = {i: id for i, id in enumerate(events_df.index)}
        id_to_index = {id: i for i, id in enumerate(events_df.index)}

        # Save FAISS index
        emb = normalized_embeddings.astype("float32")
        n, d = emb.shape
        shard_size = 15000  # adjust to keep each file <100MB

        n_shards = math.ceil(n / shard_size)

        for shard_id in range(n_shards):
            start = shard_id * shard_size
            end = min(start + shard_size, n)
            part = emb[start:end]

            idx = faiss.IndexFlatIP(d)
            idx.add(part)

            faiss.write_index(idx, f"models/event_embeddings_shard_{shard_id:02d}.index")

        # Save mappings
        meta = {
            "index_to_id": index_to_id,
            "id_to_index": id_to_index,
            "model_name": model_name_label,
        }

        with open(meta_path, "wb") as f:
            pickle.dump(meta, f)

    return events_df, user_event_associations_df, index, index_to_id, id_to_index, normalized_embeddings
    
# topic_data = await setup_topic_model()
topic_data = await setup_topic_model()

events_df, full_associations_df, index, index_to_id, id_to_index, normalized_embeddings = topic_data

Loading from 3 parquet files in 'data/events_df_cache/'...
Loading from 14 parquet files in 'data/user_event_associations_cache/'...
building indexes...


Batches: 100%|██████████| 2033/2033 [29:47<00:00,  1.14it/s]


In [18]:
# Collaborative filtering  setup
from collaborative_filtering.cf_recommender import load_cf_model

load_cf_model()

Here is our combined recommender:

In [19]:
# between 0.2 and 1.0, this ensures that even the lowest ranked item has some contribution

def normalize_scores(recommendations):
    if not recommendations:
        return []
    
    scores = [r['score'] for r in recommendations]
    min_s, max_s = min(scores), max(scores)
    
    # Range to normalize to
    LOWER_BOUND = 0.2
    UPPER_BOUND = 1.0
    
    if max_s == min_s:
        return [{'id': r['id'], 'score': UPPER_BOUND} for r in recommendations]
        
    return [
        {
            'id': r['id'], 
            'score': LOWER_BOUND + (r['score'] - min_s) * (UPPER_BOUND - LOWER_BOUND) / (max_s - min_s)
        }
        for r in recommendations
    ]


In [20]:
from topic_based_recommender.recommend_events_for_user_by_topic_based import recommend_events_for_user_by_topic_based
from recommendation_system.main import get_recommendations_for_user, get_recommendations_for_user_fast
from collaborative_filtering.cf_recommender import recommend_events
from collections import defaultdict

def get_combined_recommendations(user_address, user_train_df, top_n=10):
    weights = {'cf': 0.34, 'apriori': 0.33, 'topic': 0.33}
    
    final_scores = defaultdict(float)
    contributions = defaultdict(dict) # Store contribution details
    
    # Store individual recommendations for evaluation
    individual_recs = {
        'cf': [],
        'apriori': [],
        'topic': []
    }
    
    # Collaborative filtering
    try:
        cf_raw = recommend_events(user_address, n=20)
        if cf_raw:
            cf_recs = [{'id': str(r['id']), 'score': r['score']} for r in cf_raw]
            individual_recs['cf'] = [r['id'] for r in cf_recs[:top_n]] # Store top N IDs
            
            cf_norm = normalize_scores(cf_recs)
            for r in cf_norm:
                weighted_score = r['score'] * weights['cf']
                final_scores[r['id']] += weighted_score
                contributions[r['id']]['cf'] = {'raw': r['score'], 'weighted': weighted_score}
    except Exception as e:
        print(e)
        pass

    # Association Rules
    user_markets = []
    if not user_train_df.empty:
        user_markets = list(user_train_df['item'].dropna().unique())
    
    if user_markets:
        ap_raw = get_recommendations_for_user_fast(user_address)
        ap_recs = []
        for r in ap_raw:
            m_item = r['recommended_market']
            if m_item in item_to_event_id:
                eid = item_to_event_id[m_item]
                #  confidence as score
                ap_recs.append({'id': eid, 'score': r['confidence']})
        
        individual_recs['apriori'] = [r['id'] for r in ap_recs[:top_n]]
        
        ap_norm = normalize_scores(ap_recs)
        for r in ap_norm:
            weighted_score = r['score'] * weights['apriori']
            final_scores[r['id']] += weighted_score
            contributions[r['id']]['apriori'] = {'raw': r['score'], 'weighted': weighted_score}

    # Topic Based
    if not user_train_df.empty:
        topic_hist = user_train_df[['address', 'event_id']].dropna().drop_duplicates()
        if not topic_hist.empty:
            # We also need 'name' and 'pseudonym' cols expected by the func, even if dummy
            topic_hist['name'] = 'user'
            topic_hist['pseudonym'] = 'user'
            
            try:
                topic_raw = recommend_events_for_user_by_topic_based(
                    id_to_index, normalized_embeddings, index, index_to_id, 
                    topic_hist, events_df, 
                    user_address, top_n=20, similar_per_event=5
                )
                
                if topic_raw:
                    topic_recs = [{'id': str(r['id']), 'score': r['total_score']} for r in topic_raw]
                    individual_recs['topic'] = [r['id'] for r in topic_recs[:top_n]]
                    
                    topic_norm = normalize_scores(topic_recs)
                    for r in topic_norm:
                        weighted_score = r['score'] * weights['topic']
                        final_scores[r['id']] += weighted_score
                        contributions[r['id']]['topic'] = {'raw': r['score'], 'weighted': weighted_score}
            except Exception as e:
                pass

    # Sort and return
    results = sorted(final_scores.items(), key=lambda x: x[1], reverse=True)
    combined_recs = [{'id': i, 'score': s, 'contributions': contributions[i]} for i, s in results[:top_n]]
    
    return combined_recs, individual_recs

In [21]:

total_users = 0
k = 10

results = {
    'combined': {'precisions': [], 'hits': 0},
    'cf': {'precisions': [], 'hits': 0},
    'apriori': {'precisions': [], 'hits': 0},
    'topic': {'precisions': [], 'hits': 0}
}

# Group initial train/test data by user for fast access
train_groups = initial_train_data.groupby('address')
test_groups = initial_test_data.groupby('address')

# Only evaluate users who actually have test data
eval_users = list(test_groups.groups.keys())
print(f"Evaluating {len(eval_users)} users...")

for i, user in enumerate(eval_users):
    if i % 11 == 0:
        pct = (i + 1) / len(eval_users) * 100
        print(f"\rProcessing: {i + 1}/{len(eval_users)} ({pct:5.1f}%)", end="", flush=True)
        
    train_data = train_groups.get_group(user) if user in train_groups.groups else pd.DataFrame()
    test_data = test_groups.get_group(user)  # We know this exists
        
    # True items in test set (Event IDs)
    true_event_ids = set(test_data['event_id'].dropna().unique())
    if not true_event_ids:
        continue
    
    # Get Recommendations
    recs, ind_recs = get_combined_recommendations(user, train_data, top_n=k)

    all_algos = {
        'combined': {str(r['id']) for r in recs},
        'cf': set(map(str, ind_recs.get('cf', []))),
        'apriori': set(map(str, ind_recs.get('apriori', []))),
        'topic': set(map(str, ind_recs.get('topic', [])))
    }
    
    # Evaluate each algorithm
    for algo, recommended_ids in all_algos.items():
        n_relevant = len(recommended_ids & true_event_ids)
        n_recommended = len(recommended_ids)
        
        # Precision@k: relevant / min(k, actual recommendations)
        precision = n_relevant / min(k, n_recommended) if n_recommended > 0 else 0
        
        results[algo]['precisions'].append(precision)
        if n_relevant > 0:
            results[algo]['hits'] += 1
            
    total_users += 1

print("\n\n--- Evaluation Results ---")
print(f"Total Users Evaluated: {total_users}")
print(f"{'Algorithm':<15} | {'Precision@10':<15} | {'Hit Rate@10':<15}")
print("-" * 50)
for algo in ['combined', 'cf', 'apriori', 'topic']:
    mean_p = np.mean(results[algo]['precisions']) if results[algo]['precisions'] else 0.0
    hr = results[algo]['hits'] / total_users if total_users > 0 else 0.0
    print(f"{algo:<15} | {mean_p:.4f}          | {hr:.4f}")



Evaluating 8498 users...
Processing: 1/8498 (  0.0%)Found proxyWallet for '0x00000000000050ba7c429821e6d66429452ba168': 0x91e0f415ef4cd318a4216960118a63639264d5f4
FETCHING TRANSACTION DATA FOR SINGLE USER
Raw rows fetched for user: 0
User '0x00000000000050ba7c429821e6d66429452ba168' transaction statistics:
	Total transactions (after min_items=1): 0
Found proxyWallet for '0x000000085d9a759bb5c3d459d638739c0f48deb0': 0x2bc8bbfd8d5c787380330f448463473dabd3b39b
FETCHING TRANSACTION DATA FOR SINGLE USER
Raw rows fetched for user: 0
User '0x000000085d9a759bb5c3d459d638739c0f48deb0' transaction statistics:
	Total transactions (after min_items=1): 0
Found proxyWallet for '0x00000d1a03480d4cbfaa05048c4fae30bf409ea4': 0x680e64b067a3d012e61e07449f2b231ed86932c8
FETCHING TRANSACTION DATA FOR SINGLE USER
Raw rows fetched for user: 0
User '0x00000d1a03480d4cbfaa05048c4fae30bf409ea4' transaction statistics:
	Total transactions (after min_items=1): 0
Found proxyWallet for '0x000370bd7c2290f18838b4d42e

In [22]:
# Show example recommendation for one user
example_user = list(test_groups.groups.keys())[0]
# Group raw_trades by user for fast access
raw_trades_grouped = raw_trades.groupby('address')

if example_user:
    print(f"\nExample for User: {example_user}")
    u_data = raw_trades_grouped.get_group(example_user)
    tr_data = u_data[u_data['timestamp'] < cutoff_map[example_user]]

    recs, _ = get_combined_recommendations(example_user, tr_data, top_n=10)
    for r in recs:
        eid = r['id']
        # fetch title if possible
        title = "Unknown"
        if eid in events_df.index.astype(str):
            title = events_df.loc[int(eid)]['title']
        
        print(f"[{r['score']:.2f}] {title} ({eid})")
        
        # Print breakdown
        contribs = []
        for algo, vals in r['contributions'].items():
            # Normalized raw score (0.2-1.0)
            norm_raw = vals['raw'] 
            # Weighted contribution
            w_score = vals['weighted']
            contribs.append(f"{algo}: {w_score:.2f} (norm: {norm_raw:.2f})")
        print(f"    -> {', '.join(contribs)}")
else:
    print("No valid user found in raw_trades for example.")


Example for User: 0x00000000000050ba7c429821e6d66429452ba168
Found proxyWallet for '0x00000000000050ba7c429821e6d66429452ba168': 0x91e0f415ef4cd318a4216960118a63639264d5f4
FETCHING TRANSACTION DATA FOR SINGLE USER
Raw rows fetched for user: 0
User '0x00000000000050ba7c429821e6d66429452ba168' transaction statistics:
	Total transactions (after min_items=1): 0
[0.63] What price will $BTC reach in May? (10525)
    -> cf: 0.34 (norm: 1.00), topic: 0.29 (norm: 0.88)
[0.34] Bitcoin above $70,000 on April 12? (10270)
    -> cf: 0.34 (norm: 1.00)
[0.34] Bitcoin price 24 hours after halving? (10359)
    -> cf: 0.34 (norm: 1.00)
[0.34] Bitcoin above $60,000 on May 10? (10520)
    -> cf: 0.34 (norm: 1.00)
[0.33] Bitcoin all time high today? (903689)
    -> topic: 0.33 (norm: 1.00)
[0.32] Bitcoin all time high by March 5? (903720)
    -> topic: 0.32 (norm: 0.98)
[0.32] Bitcoin all time high by Friday? (903583)
    -> topic: 0.32 (norm: 0.97)
[0.32] Bitcoin all time high by next Friday? (903584)
  